# EXP002: Evaluation and Error Analysis

Evaluate the trained YOLOv8s 960px model on the test split and prepare predictions for error analysis.

In [4]:
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

PROJECT_PATH = Path("D:/master/Master_Drone_Detection")
EXP2_BEST_MODEL = (
    PROJECT_PATH
    / "notebooks"
    / "runs"
    / "04_experiments"
    / "EXP002_YOLOv8s_high_resolution"
    / "YOLOv8s_960-3"
    / "weights"
    / "best.pt"
)
TEST_IMAGES = PROJECT_PATH / "02_datasets" / "DUT_Anti_UAV" / "images" / "test"

assert EXP2_BEST_MODEL.exists(), f"Missing model: {EXP2_BEST_MODEL}"
assert TEST_IMAGES.exists(), f"Missing test directory: {TEST_IMAGES}"

model_exp2 = YOLO(str(EXP2_BEST_MODEL))
test_images = sorted(TEST_IMAGES.glob("*.jpg"))
print("Model:", EXP2_BEST_MODEL)
print("Test images:", len(test_images))

Model: D:\master\Master_Drone_Detection\notebooks\runs\04_experiments\EXP002_YOLOv8s_high_resolution\YOLOv8s_960-3\weights\best.pt
Test images: 2200


In [8]:
pred_records_exp2 = []

for start in range(0, len(test_images), 20):
    batch_images = test_images[start:start + 20]

    batch_results = model_exp2.predict(
        source=batch_images,
        imgsz=960,
        conf=0.25,
        iou=0.7,
        device=0,
        workers=0,
        verbose=False,
    )

    for image_path, result in zip(batch_images, batch_results):
        image_name = image_path.stem

        for pred_id, box in enumerate(result.boxes):
            xyxy = box.xyxy[0].cpu().numpy()
            pred_records_exp2.append({
                "image": image_name,
                "pred_id": pred_id,
                "x1": float(xyxy[0]),
                "y1": float(xyxy[1]),
                "x2": float(xyxy[2]),
                "y2": float(xyxy[3]),
                "confidence": float(box.conf[0]),
                "class": int(box.cls[0]),
            })

    print(f"Processed {start + len(batch_images)}/{len(test_images)}")

Processed 20/2200
Processed 40/2200
Processed 60/2200
Processed 80/2200
Processed 100/2200
Processed 120/2200
Processed 140/2200
Processed 160/2200
Processed 180/2200
Processed 200/2200
Processed 220/2200
Processed 240/2200
Processed 260/2200
Processed 280/2200
Processed 300/2200
Processed 320/2200
Processed 340/2200
Processed 360/2200
Processed 380/2200
Processed 400/2200
Processed 420/2200
Processed 440/2200
Processed 460/2200
Processed 480/2200
Processed 500/2200
Processed 520/2200
Processed 540/2200
Processed 560/2200
Processed 580/2200
Processed 600/2200
Processed 620/2200
Processed 640/2200
Processed 660/2200
Processed 680/2200
Processed 700/2200
Processed 720/2200
Processed 740/2200
Processed 760/2200
Processed 780/2200
Processed 800/2200
Processed 820/2200
Processed 840/2200
Processed 860/2200
Processed 880/2200
Processed 900/2200
Processed 920/2200
Processed 940/2200
Processed 960/2200
Processed 980/2200
Processed 1000/2200
Processed 1020/2200
Processed 1040/2200
Processed 106

In [11]:
pred_clean_exp2 = pd.DataFrame(pred_records_exp2)

print("Total predictions:", len(pred_clean_exp2))
pred_clean_exp2.head()

Total predictions: 2297


,image,pred_id,x1,y1,x2,y2,confidence,class
0,00001,0,612.680359,541.363953,682.680542,601.189331,0.910450,0
1,00002,0,511.420410,225.187347,574.551636,284.294403,0.887857,0
2,00003,0,511.598389,74.138855,773.979919,190.275635,0.897713,0
3,00004,0,780.678894,68.021729,897.213867,115.864098,0.889122,0
4,00005,0,665.319092,351.509033,690.790283,374.094482,0.757156,0


## Next analysis steps

Use `pred_clean_exp2` for IoU matching, object-size analysis, false-positive and false-negative analysis, and comparison with EXP001.

In [6]:
from PIL import Image

LABELS_DIR = PROJECT_PATH / "02_datasets" / "DUT_Anti_UAV" / "labels" / "test"


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union else 0.0


def get_gt_boxes(image_name):
    image_path = TEST_IMAGES / f"{image_name}.jpg"
    label_path = LABELS_DIR / f"{image_name}.txt"

    with Image.open(image_path) as image:
        image_width, image_height = image.size

    gt_boxes = []
    with label_path.open() as labels_file:
        for line in labels_file:
            values = line.split()
            _, x_center, y_center, width, height = map(float, values[:5])
            gt_boxes.append([
                (x_center - width / 2) * image_width,
                (y_center - height / 2) * image_height,
                (x_center + width / 2) * image_width,
                (y_center + height / 2) * image_height,
            ])

    return gt_boxes


assert LABELS_DIR.exists(), f"Missing labels directory: {LABELS_DIR}"
print("Ground-truth helpers ready")

Ground-truth helpers ready


In [12]:
if "pred_clean_exp2" not in globals():
    raise RuntimeError(
        "pred_clean_exp2 is not defined. Run the setup, prediction, and "
        "DataFrame cells above this cell first."
    )

pred_clean_exp2["best_iou"] = 0.0

for idx, row in pred_clean_exp2.iterrows():
    gt_boxes = get_gt_boxes(row.image)
    pred_box = [row.x1, row.y1, row.x2, row.y2]
    pred_clean_exp2.loc[idx, "best_iou"] = max(
        (calculate_iou(pred_box, gt_box) for gt_box in gt_boxes),
        default=0.0,
    )

print("IoU calculation completed")

IoU calculation completed


In [13]:
FP_exp2 = pred_clean_exp2[
    pred_clean_exp2.best_iou < 0.5
].copy()


background_fp_exp2 = FP_exp2[
    FP_exp2.best_iou == 0
]


localization_fp_exp2 = FP_exp2[
    (FP_exp2.best_iou > 0) &
    (FP_exp2.best_iou < 0.5)
]


print("Total FP:", len(FP_exp2))
print("Background FP:", len(background_fp_exp2))
print("Localization Error:", len(localization_fp_exp2))

Total FP: 142
Background FP: 98
Localization Error: 44


In [18]:
from pathlib import Path

LABELS = Path(
    "D:/master/Master_Drone_Detection/02_datasets/DUT_Anti_UAV/labels/val"
)

print(LABELS)

D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\labels\val


In [28]:
def analyze_localization_fixed(localization_fp):

    rows = []

    for _, row in localization_fp.iterrows():

        gt_boxes = get_gt_boxes(row.image)

        best_gt = None
        best_iou = 0


        pred_box = [
            row.x1,
            row.y1,
            row.x2,
            row.y2
        ]


        for gt in gt_boxes:

            iou = calculate_iou(
                pred_box,
                gt
            )

            if iou > best_iou:
                best_iou = iou
                best_gt = gt


        if best_gt is not None:

            gt_w = best_gt[2] - best_gt[0]
            gt_h = best_gt[3] - best_gt[1]


            pred_w = row.x2 - row.x1
            pred_h = row.y2 - row.y1


            rows.append({

                "image": row.image,

                "pred_width": pred_w,
                "gt_width": gt_w,

                "pred_height": pred_h,
                "gt_height": gt_h,

                "width_ratio": pred_w / gt_w,

                "height_ratio": pred_h / gt_h,

                "iou": best_iou
            })


    return pd.DataFrame(rows)

In [29]:
loc_df_exp2 = analyze_localization_fixed(
    localization_fp_exp2
)


print(
    "Average width ratio:",
    loc_df_exp2.width_ratio.mean()
)


print(
    "Average height ratio:",
    loc_df_exp2.height_ratio.mean()
)

Average width ratio: 1.595591968389565
Average height ratio: 1.8359619235202909
